# Lab: k-NN em vinhos e dígitos manuscritos

Notebook de aula da página *k-Vizinhos Mais Próximos*: https://insper.github.io/ML/pt/2026.2/classes/knn/

Dois conjuntos de dados que vêm com o scikit-learn. A **parte fundamental** (passos 1–5) usa o conjunto **wine** (178 vinhos, 13 medidas químicas, 3 cultivares) para abrir o k-NN por dentro: ver as unidades decidirem a distância, olhar os vizinhos de um vinho votarem, reescrever o método em quatro linhas de NumPy e escolher $k$ numa divisão de validação. A **parte avançada** (passos 6–8) passa para **dígitos manuscritos** (imagens 8×8, 64 atributos de pixel), em que dá para ver os vizinhos, literalmente, e descobrir o que a distância entende por "parecido" e o que ela não entende.

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar os vinhos e olhar as unidades

**Preveja.** Numa distância euclidiana entre dois vinhos, qual destes quatro atributos vai pesar mais: teor alcoólico, tonalidade (`hue`), magnésio ou prolina? Por quê?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target        # 13 medidas químicas; 3 cultivares (0, 1, 2)
print(X.shape)
print(y.value_counts().sort_index().to_dict())
print(X[['alcohol', 'hue', 'magnesium', 'proline']].agg(['min', 'max', 'std']).round(2))

## Passo 2 — dividir em treino, validação e teste

**Preveja.** Para que serve cada uma das três partes? Qual delas o k-NN vai guardar dentro de si no `fit`?

In [ ]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25,
                                                  stratify=y_tmp, random_state=42)
print(len(X_train), len(X_val), len(X_test))

## Passo 3 — baseline e o preço de pular o escalonador

**Preveja.** O mesmo 5-NN, com e sem o `StandardScaler`: qual acerta mais na validação? A diferença vai ser de um ou dois vinhos ou bem maior?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    'baseline (majoritária)': DummyClassifier(strategy='most_frequent'),
    'k-NN, atributos brutos': KNeighborsClassifier(n_neighbors=5),
    'k-NN, padronizado':      make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:23s} acurácia de validação {model.score(X_val, y_val):.3f}")

**Simulação: a unidade da prolina.** A prolina está em mg/L. Reescreva-a em g/L (divida a coluna por 1.000, no treino e na validação) e rode de novo o 5-NN bruto e o padronizado. São os mesmos vinhos, com a mesma informação. O que muda em cada modelo? Qual coluna passa a mandar na distância bruta?

In [ ]:
# sua simulação aqui

## Passo 4 — o k-NN por dentro: os vizinhos votam

**Preveja.** O `fit` da regressão logística aprendia um peso por atributo. O que o `fit` do k-NN guarda? Se os 5 vizinhos de um vinho são dos cultivares 0, 1, 0, 1, 0, o que o `predict_proba` devolve?

In [ ]:
knn5 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print(f"o fit guardou {knn5[-1].n_samples_fit_} linhas de {knn5[-1].n_features_in_} atributos")

wine_id = 123                                        # um vinho da validação
query = X_val.loc[[wine_id]]
dist, idx = knn5[-1].kneighbors(knn5[0].transform(query))
print(pd.DataFrame({'vizinho': X_train.index[idx[0]], 'distância': dist[0].round(2),
                    'cultivar': y_train.iloc[idx[0]].to_numpy()}).to_string(index=False))
print("votos por cultivar:", np.bincount(y_train.iloc[idx[0]], minlength=3))
print("predict_proba:", knn5.predict_proba(query)[0], "· predict:", knn5.predict(query)[0],
      "· verdade:", y_val[wine_id])

O método inteiro cabe em quatro linhas de NumPy: distâncias, ordenar, pegar os $k$ primeiros, votar.

In [ ]:
Z_train, Z_val = knn5[0].transform(X_train), knn5[0].transform(X_val)
D = np.sqrt(((Z_val[:, None, :] - Z_train[None, :, :]) ** 2).sum(axis=2))   # distância de cada vinho
nearest = np.argsort(D, axis=1)[:, :5]                                      # de validação a cada vinho
votes = y_train.to_numpy()[nearest]                                         # de treino
mine = np.array([np.bincount(v, minlength=3).argmax() for v in votes])
print(D.shape, "· igual ao scikit-learn:", (mine == knn5.predict(X_val)).all())

**Simulação: o voto muda com $k$.** Para o vinho 123, peça os 15 vizinhos mais próximos de uma vez e, para $k$ = 1, 3, 5, 7, 9, 11, 13 e 15, conte os votos dos $k$ primeiros e imprima a previsão. Com que $k$ a previsão fica certa?

In [ ]:
# sua simulação aqui

## Passo 5 — escolher $k$ na validação e testar uma vez

**Preveja.** Com $k = 1$, quanto dá a acurácia de *treino*? Esse número diz algo sobre vinhos novos?

In [ ]:
rows = []
for k in range(1, 32, 2):
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    model.fit(X_train, y_train)
    val_acc = model.score(X_val, y_val)
    rows.append({'k': k, 'treino': model.score(X_train, y_train), 'validação': val_acc,
                 'erros val': round((1 - val_acc) * len(y_val))})
table = pd.DataFrame(rows)
print(table.round(3).to_string(index=False))
best_k = int(table.loc[table['validação'].idxmax(), 'k'])   # primeiro (menor) k com o melhor escore

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
final.fit(X_tmp, y_tmp)                                      # treino + validação, mesma receita
print(f"k escolhido: {best_k} · acurácia de teste {final.score(X_test, y_test):.3f} em {len(y_test)} vinhos")

## Passo 6 — dígitos: um conjunto em que os pixels são os atributos

**Preveja.** Numa imagem 8×8, os 64 pixels já estão na mesma unidade (tinta, de 0 a 16). Padronizar vai ajudar, atrapalhar ou não mudar nada? E quais dígitos você espera que o k-NN confunda?

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
Xd, yd = digits.data, digits.target          # 1797 imagens de 8x8 pixels, cada pixel de 0 a 16
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.25,
                                                        stratify=yd, random_state=0)
print(Xd.shape, len(Xd_train), len(Xd_test))

knn_d = KNeighborsClassifier(n_neighbors=3).fit(Xd_train, yd_train)
pred = knn_d.predict(Xd_test)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)).fit(Xd_train, yd_train)
print(f"baseline             {DummyClassifier().fit(Xd_train, yd_train).score(Xd_test, yd_test):.3f}")
print(f"3-NN, pixels brutos  {(pred == yd_test).mean():.3f}")
print(f"3-NN, padronizado    {scaled.score(Xd_test, yd_test):.3f}")

wrong = np.flatnonzero(pred != yd_test)
print(f"{len(wrong)} de {len(yd_test)} dígitos de teste errados")
print(pd.crosstab(pd.Series(yd_test, name='verdadeiro'), pd.Series(pred, name='previsto')))

## Passo 7 — olhar os vizinhos

**Preveja.** Para um 9 que o modelo leu como 3, que imagens você espera encontrar entre os seus 5 vizinhos mais próximos?

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(8, 3.3))
for row, i in zip(axes, [0, wrong[0]]):                     # um certo, um errado
    dist, idx = knn_d.kneighbors(Xd_test[[i]], n_neighbors=5)
    print(f"teste #{i}: verdadeiro {yd_test[i]}, previsto {pred[i]} · vizinhos {yd_train[idx[0]]} "
          f"a distâncias {dist[0].round(1)}")
    row[0].imshow(Xd_test[i].reshape(8, 8), cmap='gray_r')
    row[0].set_title(f"verd. {yd_test[i]}, prev. {pred[i]}")
    for ax, j, d in zip(row[1:], idx[0], dist[0]):
        ax.imshow(Xd_train[j].reshape(8, 8), cmap='gray_r')
        ax.set_title(f"{yd_train[j]} (d={d:.1f})")
    for ax in row:
        ax.axis('off')
plt.tight_layout()
plt.show()

**Simulação: deslocar um dígito um pixel.** Pegue o 2 de teste #0 e desloque a imagem uma coluna para a direita (a coluna que entra fica em branco). A que distância a imagem deslocada fica da original? Compare com a distância dos vizinhos acima (20,3). O que o 3-NN prevê para ela? Depois desloque *todas* as imagens de teste 1 e 2 pixels, para os dois lados, e meça a acurácia.

In [ ]:
# sua simulação aqui

## Passo 8 — acrescentar atributos de ruído

**Preveja.** Acrescente às 64 colunas reais colunas de ruído puro, na mesma faixa de 0 a 16. Com 64 colunas de ruído (tantas quanto as reais), a acurácia cai muito ou pouco? E com 1.024?

In [ ]:
rng = np.random.default_rng(0)
for n_noise in [0, 64, 256, 1024]:
    noise = rng.uniform(0, 16, size=(len(Xd), n_noise))      # "pixels" falsos, na mesma faixa dos reais
    Xn = np.hstack([Xd, noise])
    a, b, c, d = train_test_split(Xn, yd, test_size=0.25, stratify=yd, random_state=0)
    acc = KNeighborsClassifier(n_neighbors=3).fit(a, c).score(b, d)
    print(f"{n_noise:5d} colunas de ruído ({Xn.shape[1]:4d} no total): acurácia {acc:.3f}")

**Simulação: o tamanho do ruído.** Mantenha 1.024 colunas de ruído, mas sorteie-as em faixas menores: de 0 a 16, de 0 a 4 e de 0 a 1. Para cada faixa, meça a acurácia do 3-NN sem escalonamento e com `StandardScaler` num pipeline. O que prejudica o k-NN: o número de colunas de ruído ou quanto elas somam à distância?

In [ ]:
# sua simulação aqui

**Responda.** Em uma frase cada:

1. o que o `fit` do k-NN guarda e em que momento acontece o trabalho de verdade?
2. Por que o k-NN sem escalonamento mudou de resposta quando a prolina mudou de unidade?
3. Por que um 2 deslocado de um pixel foi lido como 0?

## Sua vez

Para casa, com mais tempo:

1. **Pesos.** Repita o passo 5 com `KNeighborsClassifier(n_neighbors=k, weights='distance')`. A acurácia de treino fica em 1,000 para todo $k$? Por quê? (Dica: qual é o peso de um vizinho a distância 0?)
2. **Escolher $k$ para os dígitos.** O passo 6 usou $k = 3$ sem justificativa. Separe uma divisão de validação de dentro de `Xd_train`, avalie $k$ ímpar de 1 a 15 com `metric='euclidean'` e com `metric='manhattan'`, escolha o melhor par e só então meça o teste, uma vez.
3. **Ensinar o deslocamento.** Acrescente ao treino cópias de cada imagem deslocadas um pixel para a esquerda e para a direita (use a função `shift` da simulação do passo 7; os rótulos se repetem). Refaça a medida de acurácia nas imagens de teste deslocadas. Quanto volta? Quanto custa, em linhas guardadas e em distâncias por previsão?
4. **Resgate com PCA.** Na versão com 1.024 colunas de ruído de 0 a 16, coloque `PCA(n_components=30)` antes do k-NN num pipeline. Quanto da acurácia perdida volta? Por que a PCA conseguiria separar os pixels reais do ruído aqui? ([Redução de Dimensionalidade](https://insper.github.io/ML/pt/2026.2/classes/dimensionality-reduction/))
5. **Regressão.** Com `load_diabetes(return_X_y=True)`, compare o `KNeighborsRegressor` (escalonado, $k$ escolhido numa divisão de validação) com a `LinearRegression` num hold-out, usando $R^2$. Qual vence? Isso bate com o que a seção da maldição da dimensionalidade preveria com 10 atributos e 442 linhas?